In [ ]:
!pip install opencv-python-headless numpy openslide-python scikit-image torch torchvision pyyaml albumentations Pillow openslide-bin

In [ ]:
import cv2

import numpy as np
import os
import random
import skimage.color as sk_color
import skimage.io as sk_io
import sys
import torch
import torch.nn as nn
import torchvision.transforms.functional as TF
import yaml

from albumentations import ElasticTransform, GridDistortion, OpticalDistortion
from PIL import Image, ImageDraw
from torchvision import transforms

current_path = os.path.abspath('.')
root_path = os.path.dirname(os.path.dirname(current_path))
sys.path.append(root_path)

In [ ]:
def load_np_image(path, color_model="RGB"):

    pil_img = load_pil_image(path, gray=color_model == "GRAY", color_model=color_model)
    return pil_to_np(pil_img)


def load_pil_image(path, gray=False):

    with open(path, 'rb') as f:

        if gray:
            return Image.open(f).convert('L')     # grayscale

        return Image.open(f).convert('RGB')    # rgb

def pil_to_np(pil_img):
    """
    Convert a PIL Image to a NumPy array.
    Note that RGB PIL (w, h) -> NumPy (h, w, 3).
    Args:
    pil_img: The PIL Image.
    Returns:
    The PIL image converted to a NumPy array.
    """

    rgb = np.asarray(pil_img)
    return rgb


def np_to_pil(np_img):
    """
    Convert a NumPy array to a PIL Image.
    Args:
        np_img: The image represented as a NumPy array.
    Returns:
    The NumPy array converted to a PIL Image.
    """

    if np_img.dtype == "bool":
        np_img = np_img.astype("uint8") * 255
    return Image.fromarray(np_img)


In [ ]:
def data_augmentation(input_image, target_img, img_input_size=(640, 640), img_output_size=(640, 640), aug=None, GAN_model=None):

    image = TF.resize(input_image, size=img_output_size)
    target_image = TF.resize(target_img, size=img_output_size) if target_img is not None else None

    used_augmentations = []
    if aug is not None and len(aug) > 0:

        # Random horizontal flipping
        if "horizontal_flip" in aug and (len(aug) < 2 or random.random() > 0.5):
            image = TF.hflip(image)
            used_augmentations.append("horizontal_flip")

        # Random vertical flipping
        if "vertical_flip" in aug and (len(aug) < 2 or random.random() > 0.5):
            image = TF.vflip(image)
            used_augmentations.append("vertical_flip")

        # Random rotation
        if "rotation" in aug and (len(aug) < 2 or random.random() > 0.5):
            angle = random.uniform(-25, 25) 
            image = TF.rotate(image, angle, expand=False, fill=0)
            used_augmentations.append("rotation")

        # Random elastic transformation
        if "elastic_transformation" in aug and (len(aug) < 2 or random.random() > 0.5):
            alpha = random.randint(100, 200)
            augmented = ElasticTransform(p=1, alpha=alpha, sigma=alpha * 0.05, alpha_affine=alpha * 0.03)(image=np.array(image))
            image = Image.fromarray(augmented['image'])
            used_augmentations.append("elastic_transformation")

        # Random GridDistortion
        if "grid_distortion" in aug and (len(aug) < 2 or random.random() > 0.5):
            augmented = GridDistortion(p=1)(image=np.array(image))
            image = Image.fromarray(augmented['image'])
            used_augmentations.append("grid_distortion")

        # Random OpticalDistortion
        if "optical_distortion" in aug and (len(aug) < 2 or random.random() > 0.5):
            augmented = OpticalDistortion(p=1, distort_limit=1, shift_limit=0.5)(image=np.array(image))
            image = Image.fromarray(augmented['image'])
            used_augmentations.append("optical_distortion")

    return image, used_augmentations

In [ ]:
# Código adaptado -> gerar imagens sem considerar as márcaras 
def process_dataset(input_root, output_root, augmentation, aumentos, gan_model=None):
    for class_name in os.listdir(input_root):
        class_path = os.path.join(input_root, class_name)
        if not os.path.isdir(class_path):
            continue

        output_class_path = os.path.join(output_root, class_name)
        os.makedirs(output_class_path, exist_ok=True)

        for img_name in os.listdir(class_path):
            if not img_name.lower().endswith(('.jpg', '.jpeg', '.png', '.tif')):
                continue

            for aumento_idx in range(aumentos):
                print(f"Aumento {aumento_idx + 1} para imagem {img_name}")

                img_path = os.path.join(class_path, img_name)
                input_image = load_pil_image(img_path, False)

                target_img_name = random.choice([x for x in os.listdir(class_path) if x.lower().endswith(('.jpg', '.jpeg', '.png', '.tif'))])
                target_img_path = os.path.join(class_path, target_img_name)
                target_image = load_pil_image(target_img_path, False) if 'color_transfer' in augmentation else None

                augmented_img, _, used_augmentations = data_augmentation(
                    input_image, 
                    target_image, 
                    img_input_size=(250, 450),
                    img_output_size=(250, 450),
                    aug=augmentation, 
                    GAN_model=gan_model
                )

                # Gera nome único para cada aumento
                img_base, img_ext = os.path.splitext(img_name)
                aug_img_filename = f"aug_{img_base}_aug{aumento_idx+1}{img_ext}"
                output_img_path = os.path.join(output_class_path, aug_img_filename)

                if isinstance(augmented_img, torch.Tensor):
                    if augmented_img.dim() == 4:
                        augmented_img = augmented_img[0]
                    img_np = augmented_img.permute(1, 2, 0).cpu().numpy()
                    img_np = (img_np * 255).astype(np.uint8)
                    Image.fromarray(img_np).save(output_img_path)
                elif isinstance(augmented_img, np.ndarray):
                    Image.fromarray(augmented_img).save(output_img_path)
                elif isinstance(augmented_img, Image.Image):
                    augmented_img.save(output_img_path)
                else:
                    raise ValueError(f"Tipo de imagem não suportado: {type(augmented_img)}")

                print(f"Imagem aumentada salva: {output_img_path}")

In [ ]:

if __name__ == "__main__":
    dataset_path = "Originals" # Colocar aqui o path do dataset
    output_path = "Novos Testes/13x_Et-Gd"  # Colocar aqui o path onde as imagens aumentadas serão enviadas  
    augmentation = [ "elastic_transformation", "grid_distortion"]
    
    os.makedirs(output_path, exist_ok=True)
    

    process_dataset(dataset_path, output_path, augmentation, 13)